In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive/deeplearning/celeb_faces_regconition/

/content/drive/MyDrive/deeplearning/celeb_faces_regconition


In [3]:
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from src.datasets.dataset import get_data_loader
from src.engine.checkpoint_manager import CheckpointManager
from src.engine.trainer import Trainer
from src.engine.tracker import Tracker
from src.callbacks.early_stopping import EarlyStopping
from src.callbacks.progressive_unfreezer import ProgressiveUnfreezer
from src.utils.device import get_device
from src.utils.seed import set_seed
from models.resnet50.model.model import Model
from models.resnet50.config import cfg

In [4]:
set_seed (cfg.seed)

In [5]:
device = get_device (cfg.prefer_devices)
print (device)

cuda


In [6]:
cfg.experiment_dir.mkdir (parents = True, exist_ok = True)

In [7]:
train_loader, val_loader, test_loader = get_data_loader (cfg.data_root, cfg.batch_size, cfg.image_size, shuffle = True)

In [8]:
num_classes = len (train_loader.dataset.classes)

In [9]:
model = Model (num_classes)

In [10]:
criterion = nn.CrossEntropyLoss ()

In [11]:
optimizer = AdamW (model.parameters (), lr = cfg.learning_rate)

In [12]:
scheduler = ReduceLROnPlateau (optimizer, mode = "max", factor = 0.5, patience = 2)

In [13]:
checkpoint_manager = CheckpointManager (cfg.experiment_dir)

In [14]:
tracker = Tracker (cfg.experiment_dir)

In [15]:
callbacks = [EarlyStopping (patience=cfg.patience), ProgressiveUnfreezer(model.backbone, cfg.unfreeze_schedule)]

In [16]:
trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        scheduler=scheduler,
        tracker=tracker,
        checkpoint_manager=checkpoint_manager,
        callbacks=callbacks
    )

In [17]:
start_epoch = trainer.resume () if cfg.resume else 0
trainer.fit (start_epoch, cfg.epochs)

Epoch 0/30
Loss: 3.0954


NameError: name 'val_acc' is not defined